# DiTFlow benchmark - Colab runner

Probe, pilot, full run, download. Every step is resumable: `sweep.py` writes
`done.json` only after verifying a cell's output on disk, so a recycled runtime
loses at most the cell that was in flight. Re-run the same cell to continue.

**Before you start:**

| What | Where | Why there |
|---|---|---|
| the repo (`benchmark/`, `motion_guidance.py`, `configs/`) | Drive: `MyDrive/ditflow/repo` | small, and it changes as you work |
| `bench/packed/` | a private HF dataset repo | pulled with `allow_patterns`, so the archival raw DAVIS and MiraData stay put |
| results | Drive: `MyDrive/ditflow/runs` | written as the sweep goes, survives a recycled runtime |


## 1. Runtime


In [1]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB

torch 2.11.0+cu128 | cuda True


## 2. Secrets, repo, dependencies

Tokens live on Drive, never in the notebook: `MyDrive/secrets/gh_token.env` and
`hf_token.env`, each one `KEY = value` line. A notebook gets shared or committed
eventually, and a pasted token then has to be revoked.

`diffusers==0.30.2` is pinned by requirements.txt as REQUIRED. The API surface
DiTFlow uses is unchanged in 0.36, so the newer one is worth trying first if the
pin fights your torch build - but if the probe errors inside the transformer or
produces garbage, pin 0.30.2 before assuming the method is at fault.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/ditflow'
RUNS = DRIVE + '/runs'   # on Drive, so a recycled runtime keeps its results

# grid.yaml = CogVideoX-2B, grid_5b.yaml = 5B (A100), grid_wan.yaml = Wan2.1-14B.
# Each carries a run_tag, so the three write to separate trees and never
# overwrite one another.
GRID = 'benchmark/grid_5b.yaml'
GH_REPO = 'John00451/ditflow'      # <- your private repo


def load_env(path):
    """Parse `KEY = value` lines; strips quotes and trailing comments."""
    out = {}
    for line in open(path, encoding='utf-8'):
        line = line.split('#', 1)[0].strip()
        if '=' in line:
            k, v = line.split('=', 1)
            out[k.strip()] = v.strip().strip('"').strip("'")
    return out


SECRETS = '/content/drive/MyDrive/secrets'
GH_TOKEN = load_env(SECRETS + '/gh_token.env')['GH_TOKEN']
HF_TOKEN = load_env(SECRETS + '/hf_token.env')['HF_TOKEN']
print('tokens loaded:', GH_TOKEN[:4] + '...', HF_TOKEN[:4] + '...')


Mounted at /content/drive


In [ ]:
import os
import subprocess

REPO_DIR = '/content/ditflow'
url = f'https://{GH_TOKEN}@github.com/{GH_REPO}.git'

if os.path.isdir(REPO_DIR + '/.git'):
    # The token goes into .git/config for the length of the pull and is
    # stripped again below, so a zipped or shared runtime does not carry a
    # working credential out with it.
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', url, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin',
                f'https://github.com/{GH_REPO}.git'], check=True)

%cd {REPO_DIR}
!git log --oneline -1
!ls benchmark/


In [ ]:
!pip install -q diffusers==0.30.2 transformers accelerate sentencepiece                imageio imageio-ffmpeg omegaconf einops
import diffusers
print('diffusers', diffusers.__version__)


## 3. Data and manifests

Only `bench/packed` is pulled. The raw `DAVIS/` (12,422 files) and
`MiraData 9K/` in the repo are archival - the sweep reads the 24-frame windows,
not the sources - and skipping them saves roughly 900 MB per session.

The manifests carry Windows absolute paths, which is deliberate: they are what
make a row reproducible. `remap.py` rewrites only the `*_path` columns, so the
recorded `source` and `sha256` still describe the same bytes. Nothing is copied
or extracted - the manifests are pointed straight at the download.


In [ ]:
from huggingface_hub import login, snapshot_download

HF_REPO = 'John00451/testDataset'
login(token=HF_TOKEN, add_to_git_credential=False)

DATA = snapshot_download(HF_REPO, repo_type='dataset', local_dir='/content/hf',
                         allow_patterns=['bench/packed/**']) + '/bench/packed'
!ls {DATA}

import subprocess
import sys

WIN = r'E:\bench\packed'
for name in ('davis50', 'miradata'):
    subprocess.run([sys.executable, 'benchmark/remap.py',
                    '--manifest', f'benchmark/{name}.csv',
                    '--out', f'/content/{name}.csv',
                    '--from', WIN, '--to', DATA], check=True)


## 4. Probe - one cell

Answers the two things every estimate depends on: does CogVideoX-2B fit this
GPU, and how long is one generation actually?


In [ ]:
!python benchmark/sweep.py --manifest /content/davis50.csv --out {RUNS} --grid {GRID} \
    --only-prompt subject --limit 1 --prune-embeds


In [ ]:
import glob
import sys
sys.path.insert(0, '/content/ditflow')
from colab_utils import show_run, summarize_run

cell = sorted(glob.glob(RUNS + '/davis/*/subject/*/seed*'))[0]
summarize_run(cell)
show_run(cell)


## 5. Pilot - a few clips, every config

Six configs on three clips shows whether the *methods* differ, before you spend
hours confirming it at N=50. If `backbone` and `ditflow_z` come out looking
identical here, the wiring is wrong, not the science.


In [ ]:
import csv

ids = [r['clip_id'] for r in csv.DictReader(open('/content/davis50.csv'))
       if r['prompt_id'] == 'subject'][:3]
sel = ' '.join('--only-clip ' + i for i in ids)
print('pilot clips:', ids)

!python benchmark/sweep.py --manifest /content/davis50.csv --out {RUNS} --grid {GRID} \
    --only-prompt subject {sel} --prune-embeds


In [ ]:
for cell in sorted(glob.glob(RUNS + '/davis/' + ids[0] + '/subject/*/seed*')):
    print(cell.split('/')[-2])
    display(show_run(cell))


## 6. Full run

Finished cells are skipped, so interrupting this costs nothing. If the runtime
dies, re-execute the same cell.


In [ ]:
!python benchmark/sweep.py --manifest /content/davis50.csv --out {RUNS} --grid {GRID} \
    --only-prompt subject --prune-embeds


In [ ]:
!python benchmark/sweep.py --manifest /content/miradata.csv --out {RUNS} --grid {GRID} \
    --only-prompt subject --prune-embeds \
    --only-config ditflow_z --only-config smm --only-config backbone


## 7. Download

The results are already on Drive. This is for pulling them down as one file:
the sweep is hundreds of directories and `files.download` takes a single path,
so each cell is flattened to a name that still says which cell it came from.


In [ ]:
from colab_utils import zip_sweep
zip_sweep(RUNS, zip_path='/content/sweep.zip')


## 8. Where things stand

`--dry-run` reports progress without running anything. Failed cells keep a
`log.txt` and a `failed.json` naming the reason; add `--retry-failed` once the
cause is fixed.


In [ ]:
!python benchmark/sweep.py --manifest /content/davis50.csv --out {RUNS} --grid {GRID} \
    --only-prompt subject --dry-run | head -3
